# Econometrics Lab — The Linear Model: Tests

**Sorbonne Data Analytics — November 2025**

In this lab, we will practice the key concepts from the lecture on hypothesis testing in the linear regression model. We will work through:

1. **Refresher on distributions** (Normal, Chi-squared, Student, Fisher)
2. **The T-test** for individual coefficients
3. **Size, power, and p-values**
4. **The F-test** for joint significance
5. **Testing linear restrictions** (Cobb-Douglas example)
6. **The Chow test** for structural change

> **Note:** The Python code is kept as simple as possible. Focus on understanding the *statistics*, not the code.


---
## Setup

Run the cell below to import the libraries we will use throughout the lab.


In [ ]:
# Run this cell first
import numpy as np
import pandas as pd
import statsmodels.api as sm

# For reproducibility
np.random.seed(42)

print("All libraries loaded successfully!")

---
## Part 1 — Reminder: Common Distributions

Before running tests, we need to be comfortable with the four key distributions:
- **Normal** $N(\mu, \sigma^2)$
- **Chi-squared** $\chi^2_n$ (sum of $n$ squared standard normals)
- **Student** $T_n$ (ratio of a normal to the square root of a $\chi^2_n / n$)
- **Fisher** $F_{p,q}$ (ratio of two independent $\chi^2$ divided by their degrees of freedom)

**Key relationships to remember:**
- When $n \to \infty$, the Student $T_n$ converges to $N(0,1)$
- The square of a $T_1$ variable is an $F_{1,1}$ variable


### Exercise 1.1 — Visualizing the Student distribution

Using `scipy.stats`, plot the Student-$t$ density for several degrees of freedom ($\nu$ = 2, 5, 30) on the same graph, together with the standard Normal $N(0,1)$.

**What do you observe as $\nu$ increases?**


In [ ]:
# YOUR CODE HERE
# Hint: use np.linspace(-4, 4, 300) for the x-axis
# Hint: stats.t.pdf(x, df=...) gives the Student density
# Hint: stats.norm.pdf(x) gives the standard Normal density



### Exercise 1.2 — Critical values

Using `scipy.stats`, find:

1. The 2.5% critical value (two-tailed, $\alpha = 5\%$) of a $T_{25}$ distribution.
2. The 2.5% critical value of a $N(0,1)$ distribution.
3. The 5% critical value (right-tail) of an $F_{3, 24}$ distribution.

**Compare the Student and Normal critical values. What do you notice?**


In [ ]:
# YOUR CODE HERE
# Hint: stats.t.ppf(0.975, df=25) gives the 97.5th percentile of T_25
# Hint: stats.norm.ppf(0.975) gives the 97.5th percentile of N(0,1)
# Hint: stats.f.ppf(0.95, dfn=3, dfd=24) gives the 95th percentile of F(3,24)



---
## Part 2 — OLS Estimation and the T-test

We will simulate a simple dataset and run OLS regressions to practice T-tests.

**The model:** We generate data from the true model:

$$y_i = 2 + 3 \cdot x_{1i} + 0 \cdot x_{2i} + u_i, \quad u_i \sim N(0, 4)$$

Note that $x_2$ has **no effect** on $y$ (its true coefficient is 0). The T-test should detect this.


### Exercise 2.1 — Generate the data

Generate a dataset with $N = 50$ observations:
- $x_1$: drawn from $N(0, 1)$
- $x_2$: drawn from $N(0, 1)$
- $u$: drawn from $N(0, 4)$ (i.e., $\sigma^2 = 4$, so $\sigma = 2$)
- $y = 2 + 3 x_1 + 0 x_2 + u$


In [ ]:
N = 50

# Generate random variables
x1 = np.random.normal(0, 1, N)
x2 = np.random.normal(0, 1, N)
u = np.random.normal(0, 2, N)  # sigma = 2, so sigma^2 = 4

# True model: y = 2 + 3*x1 + 0*x2 + u
y = 2 + 3 * x1 + 0 * x2 + u

# Put in a DataFrame for convenience
data = pd.DataFrame({"y": y, "x1": x1, "x2": x2})
print(data.head(10))
print(f"\nNumber of observations: {len(data)}")

### Exercise 2.2 — Run OLS and read the output

Estimate the model $y = b_0 + b_1 x_1 + b_2 x_2 + u$ using `statsmodels`.

Then answer the following questions by reading the regression output:

1. What are the estimated coefficients $\hat{b}_0$, $\hat{b}_1$, $\hat{b}_2$?
2. What are their standard errors?
3. What is the t-statistic for each coefficient? How is it computed?
4. For each coefficient, is $H_0: b_j = 0$ rejected at the 5% level? Use the p-value.
5. Is the variable $x_2$ significant? Is this consistent with the true model?


In [ ]:
# YOUR CODE HERE
# Step 1: Add a constant (intercept) to the X matrix
# Hint: X = sm.add_constant(data[['x1', 'x2']])

# Step 2: Fit OLS
# Hint: model = sm.OLS(data['y'], X).fit()

# Step 3: Print the summary
# Hint: print(model.summary())



### Exercise 2.3 — Manual T-test computation

**Without using the p-values from statsmodels**, manually compute the t-statistic for $b_2$ and decide whether to reject $H_0: b_2 = 0$ at the 5% level.

Recall the formula:
$$t_j = \frac{\hat{b}_j - b_j^{H_0}}{\hat{\sigma}_{\hat{b}_j}} \hookrightarrow T_{N-k}$$

Here $b_j^{H_0} = 0$, $N = 50$, $k = 3$ (constant + 2 variables).

Compare $|t|$ with the critical value from $T_{47}$ at 2.5%.


In [ ]:
# YOUR CODE HERE
# Step 1: Extract b2_hat and its standard error from the model
# Hint: b2_hat = model.params['x2']
# Hint: se_b2 = model.bse['x2']

# Step 2: Compute t-statistic manually
# Hint: t_stat = (b2_hat - 0) / se_b2

# Step 3: Find the critical value from T_{N-k} = T_{47}
# Hint: t_critical = stats.t.ppf(0.975, df=N-3)

# Step 4: Compare |t| with t_critical and conclude



### Exercise 2.4 — Testing $b_1 = 3$ (the true value)

Now test whether $b_1$ is significantly different from its **true value** $b_1 = 3$.

$$H_0: b_1 = 3 \quad \text{vs.} \quad H_1: b_1 \neq 3$$

Compute the t-statistic: $t = \frac{\hat{b}_1 - 3}{\hat{\sigma}_{\hat{b}_1}}$

**Should we reject $H_0$?** (We expect not to, since 3 is the true value.)


In [ ]:
# YOUR CODE HERE
# Step 1: Extract b1_hat and its standard error

# Step 2: Compute t = (b1_hat - 3) / se_b1

# Step 3: Compute the p-value (two-tailed)
# Hint: p_value = 2 * stats.t.sf(abs(t_stat), df=N-3)

# Step 4: Conclude



---
## Part 3 — Size, Power, and the P-value

Recall:
- **Type I error** ($\alpha$): Rejecting $H_0$ when it is true (= size of the test)
- **Type II error** ($\beta$): Not rejecting $H_0$ when $H_1$ is true
- **Power** = $1 - \beta$: Probability of correctly rejecting a false $H_0$
- **P-value**: The probability, under $H_0$, of observing a test statistic as extreme or more extreme than the one computed

**Rule:** If p-value $< \alpha$, reject $H_0$.


### Exercise 3.1 — Simulating the size of a test

We will verify that the T-test rejects $H_0: b_2 = 0$ about 5% of the time **when $H_0$ is true** (since the true $b_2 = 0$).

**Procedure:**
1. Simulate 1000 datasets from the true model $y = 2 + 3x_1 + 0 x_2 + u$
2. For each dataset, run OLS and check if $H_0: b_2 = 0$ is rejected at 5%
3. Count the rejection rate — it should be close to 5%


In [ ]:
# YOUR CODE HERE
n_simulations = 1000
N = 50
rejections = 0

for i in range(n_simulations):
    # Generate data
    x1 = np.random.normal(0, 1, N)
    x2 = np.random.normal(0, 1, N)
    u = np.random.normal(0, 2, N)
    y = 2 + 3 * x1 + 0 * x2 + u

    # Run OLS
    X = sm.add_constant(np.column_stack([x1, x2]))
    model = sm.OLS(y, X).fit()

    # Check if p-value for x2 < 0.05
    # Hint: model.pvalues[2] is the p-value for x2 (index 0=const, 1=x1, 2=x2)
    # YOUR CODE: if model.pvalues[2] < 0.05: rejections += 1

# Print the rejection rate
# YOUR CODE: print(f"Rejection rate: {rejections / n_simulations:.3f}")

### Exercise 3.2 — Simulating the power of a test

Now let's see what happens when $H_0$ is **false**. Repeat the simulation but with the true model:

$$y = 2 + 3 x_1 + 0.5 x_2 + u$$

Now $b_2 = 0.5 \neq 0$, so $H_0: b_2 = 0$ is false. How often do we reject?

**Try with $N = 50$ and then $N = 200$. What happens to the power?**


In [ ]:
# YOUR CODE HERE
# Same structure as above, but change the true model to include b2 = 0.5
# Try with N = 50 and N = 200



---
## Part 4 — The F-test for Joint Significance

The T-test checks one coefficient at a time. The **F-test** checks multiple coefficients simultaneously.

**Why not just use multiple T-tests?** As shown in the lecture, if you run 20 individual tests at 5%, the probability of at least one false rejection is:
$$\alpha_{\text{global}} = 1 - (1-0.05)^{20} \approx 0.64$$

That's a 64% chance of making at least one mistake!

The F-test controls this by testing all constraints jointly.


### Exercise 4.1 — The global F-test (overall significance)

The **global F-test** reported by statsmodels tests:
$$H_0: b_1 = b_2 = 0 \quad \text{(all slope coefficients are zero)}$$

Using the regression from Part 2:
1. Read the F-statistic and its p-value from the summary.
2. Should you reject $H_0$?
3. Verify the F-statistic manually using the formula:

$$F = \frac{R^2 / (k-1)}{(1 - R^2) / (N - k)} \hookrightarrow F_{k-1, N-k}$$

where $k$ is the total number of parameters (including the constant).


In [ ]:
# First, re-estimate the model (in case you didn't save it above)
X = sm.add_constant(data[["x1", "x2"]])
model = sm.OLS(data["y"], X).fit()
print(model.summary())

In [ ]:
# YOUR CODE HERE — Verify the F-statistic manually
# Step 1: Get R-squared from the model
# Hint: R2 = model.rsquared

# Step 2: Compute F manually
# k = 3 (constant + x1 + x2)
# F = (R2 / (k-1)) / ((1 - R2) / (N - k))

# Step 3: Compare with model.fvalue



### Exercise 4.2 — Testing a subset of coefficients

Now we will generate a model with 4 regressors:

$$y = 1 + 2x_1 + 3x_2 + 0 \cdot x_3 + 0 \cdot x_4 + u$$

We want to test:
$$H_0: b_3 = 0 \text{ and } b_4 = 0 \quad \text{(jointly)}$$

This is an F-test with $q = 2$ constraints.

Use the formula based on $R^2$:
$$F = \frac{(R^2_{nc} - R^2_c)}{1 - R^2_{nc}} \cdot \frac{N - k}{q} \hookrightarrow F_{q, N-k}$$

where:
- $R^2_{nc}$: R-squared of the full (non-constrained) model with $x_1, x_2, x_3, x_4$
- $R^2_c$: R-squared of the restricted (constrained) model with only $x_1, x_2$


In [ ]:
# Generate data
N = 100
x1 = np.random.normal(0, 1, N)
x2 = np.random.normal(0, 1, N)
x3 = np.random.normal(0, 1, N)
x4 = np.random.normal(0, 1, N)
u = np.random.normal(0, 2, N)
y = 1 + 2 * x1 + 3 * x2 + 0 * x3 + 0 * x4 + u

# YOUR CODE HERE
# Step 1: Estimate the FULL model (non-constrained) with x1, x2, x3, x4

# Step 2: Estimate the RESTRICTED model (constrained) with x1, x2 only

# Step 3: Compute the F-statistic using the R² formula

# Step 4: Compute the p-value
# Hint: p_value = 1 - stats.f.cdf(F_stat, dfn=q, dfd=N-k)

# Step 5: Conclude

### Exercise 4.3 — Using statsmodels `f_test()`

Statsmodels can run F-tests directly with the `f_test()` method. 

Test the same hypothesis $H_0: b_3 = 0$ and $b_4 = 0$ using `f_test()`.


In [ ]:
# YOUR CODE HERE
# Re-estimate the full model
X_full = sm.add_constant(np.column_stack([x1, x2, x3, x4]))
model_full = sm.OLS(y, X_full).fit()

# Use f_test with a restriction matrix
# The restriction is: b3 = 0 AND b4 = 0
# In matrix form: R * b = r
# R = [[0, 0, 0, 1, 0],   (selects b3)
#      [0, 0, 0, 0, 1]]   (selects b4)
# r = [0, 0]

# Hint: result = model_full.f_test("x3 = 0, x4 = 0")  -- but we used numeric columns
# With numeric columns, use a matrix:
# R = np.array([[0, 0, 0, 1, 0],
#               [0, 0, 0, 0, 1]])
# result = model_full.f_test(R)
# print(result)

---
## Part 5 — Application: Cobb-Douglas Production Function

We use a dataset of 27 firms to estimate a Cobb-Douglas production function:

$$\log(y_i) = \log(A) + \alpha \log(K_i) + \beta \log(L_i) + u_i$$

where:
- $y$: output
- $K$: capital
- $L$: labor
- $\alpha$, $\beta$: output elasticities

We want to test:
1. Are $\alpha$ and $\beta$ individually significant?
2. **Constant returns to scale:** $H_0: \alpha + \beta = 1$ (T-test on a linear combination)
3. **Equal elasticities:** $H_0: \alpha = \beta$ (T-test on a linear combination)


In [ ]:
# Simulated Cobb-Douglas data (inspired by Greene)
np.random.seed(123)
N = 27

log_K = np.random.normal(5, 1, N)
log_L = np.random.normal(4, 0.8, N)
u = np.random.normal(0, 0.3, N)

# True parameters: log(A) = 1.5, alpha = 0.35, beta = 0.60
log_y = 1.5 + 0.35 * log_K + 0.60 * log_L + u

data_cd = pd.DataFrame({"log_y": log_y, "log_K": log_K, "log_L": log_L})
print(data_cd.head(10))

### Exercise 5.1 — Estimate the Cobb-Douglas model

Estimate the model and check if $\alpha$ and $\beta$ are individually significant.


In [ ]:
# YOUR CODE HERE



### Exercise 5.2 — Test for constant returns to scale

Test $H_0: \alpha + \beta = 1$ using a **T-test**.

Recall:
$$t = \frac{(\hat{\alpha} + \hat{\beta}) - 1}{\hat{\sigma}_{\hat{\alpha} + \hat{\beta}}}$$

You need the standard error of $\hat{\alpha} + \hat{\beta}$:
$$\text{Var}(\hat{\alpha} + \hat{\beta}) = \text{Var}(\hat{\alpha}) + \text{Var}(\hat{\beta}) + 2 \text{Cov}(\hat{\alpha}, \hat{\beta})$$

The covariance matrix is available from `model.cov_params()`.


In [ ]:
# YOUR CODE HERE
# Step 1: Get the estimated coefficients

# Step 2: Get the covariance matrix
# Hint: cov = model_cd.cov_params()
# cov.iloc[1,1] = Var(alpha_hat), cov.iloc[2,2] = Var(beta_hat), cov.iloc[1,2] = Cov

# Step 3: Compute Var(alpha_hat + beta_hat) and then the standard error

# Step 4: Compute t = (alpha_hat + beta_hat - 1) / se

# Step 5: Compute p-value and conclude



### Exercise 5.3 — Test for equal elasticities

Test $H_0: \alpha = \beta$ (equivalently, $\alpha - \beta = 0$) using the same approach.


In [ ]:
# YOUR CODE HERE
# Same approach: t = (alpha_hat - beta_hat) / se(alpha_hat - beta_hat)
# Var(alpha_hat - beta_hat) = Var(alpha_hat) + Var(beta_hat) - 2*Cov(alpha_hat, beta_hat)



---
## Part 6 — Testing for Structural Change: The Chow Test

The Chow test checks whether regression coefficients are the **same** across two subsamples.

**Idea:** If the model is stable, the constrained model (pooled) should fit almost as well as two separate regressions.

$$F = \frac{(SSR_c - SSR_1 - SSR_2) / k}{(SSR_1 + SSR_2) / (N - 2k)} \hookrightarrow F_{k, N-2k}$$

We will simulate a dataset where a structural break occurs.


In [ ]:
# Generate data with a structural break
np.random.seed(42)

# Subsample 1: y = 1 + 2*x + u  (observations 1 to 60)
N1 = 60
x1 = np.random.normal(3, 1, N1)
u1 = np.random.normal(0, 1, N1)
y1 = 1 + 2 * x1 + u1

# Subsample 2: y = 5 + 0.5*x + u  (observations 61 to 100) — DIFFERENT coefficients!
N2 = 40
x2 = np.random.normal(3, 1, N2)
u2 = np.random.normal(0, 1, N2)
y2 = 5 + 0.5 * x2 + u2

# Combined (pooled) sample
x_all = np.concatenate([x1, x2])
y_all = np.concatenate([y1, y2])

print(f"Subsample 1: {N1} observations (true model: y = 1 + 2x)")
print(f"Subsample 2: {N2} observations (true model: y = 5 + 0.5x)")
print(f"Total: {N1 + N2} observations")

### Exercise 6.1 — Visualize the structural break

Plot $y$ against $x$ for both subsamples (use different colors). Do you see the break?


In [ ]:
# YOUR CODE HERE
# Hint: plt.scatter(x1, y1, color='blue', label='Subsample 1', alpha=0.6)
# Hint: plt.scatter(x2, y2, color='red', label='Subsample 2', alpha=0.6)



### Exercise 6.2 — Run the Chow test (Method 1: three regressions)

1. Estimate the **pooled (constrained)** model on all $N = 100$ observations.
2. Estimate the model on **subsample 1** only.
3. Estimate the model on **subsample 2** only.
4. Compute the Chow F-statistic.
5. Find the p-value and conclude.

$$F = \frac{(SSR_c - SSR_1 - SSR_2) / k}{(SSR_1 + SSR_2) / (N - 2k)}$$

where $k = 2$ (constant + one slope), $N = 100$.


In [ ]:
# YOUR CODE HERE

# Step 1: Pooled model
# Hint: X_all = sm.add_constant(x_all)
# model_pooled = sm.OLS(y_all, X_all).fit()
# SSR_c = model_pooled.ssr   (ssr = sum of squared residuals)

# Step 2: Subsample 1
# X1 = sm.add_constant(x1)
# model1 = sm.OLS(y1, X1).fit()
# SSR_1 = model1.ssr

# Step 3: Subsample 2

# Step 4: Compute F
# k = 2
# N = N1 + N2
# F = ((SSR_c - SSR_1 - SSR_2) / k) / ((SSR_1 + SSR_2) / (N - 2*k))

# Step 5: P-value
# p_value = 1 - stats.f.cdf(F, dfn=k, dfd=N - 2*k)



### Exercise 6.3 — Chow test (Method 2: interacted variables)

Run the Chow test using a **single regression** with interaction terms.

Create a dummy variable $I_i = 1$ if observation $i$ belongs to subsample 1, and 0 otherwise.

Then estimate:
$$y_i = b_0 + b_1 x_i + a_0 I_i + a_1 (x_i \cdot I_i) + u_i$$

The Chow test is equivalent to testing $H_0: a_0 = 0$ and $a_1 = 0$ (F-test).


In [ ]:
# YOUR CODE HERE

# Step 1: Create the dummy I (1 for subsample 1, 0 for subsample 2)
I = np.concatenate([np.ones(N1), np.zeros(N2)])

# Step 2: Create the interaction x * I
x_I = x_all * I

# Step 3: Estimate the augmented model
# X_aug = sm.add_constant(np.column_stack([x_all, I, x_I]))
# model_aug = sm.OLS(y_all, X_aug).fit()
# print(model_aug.summary())

# Step 4: F-test for a0 = 0 and a1 = 0
# R = np.array([[0, 0, 1, 0],
#               [0, 0, 0, 1]])
# result = model_aug.f_test(R)
# print(result)

### Exercise 6.4 — Compare the two methods

Compare the F-statistic and p-value from the two methods. Are they the same (up to rounding)?


In [ ]:
# YOUR CODE HERE — just print and compare the two F-statistics



---
## Part 7 — Forecasting

From the lecture, the forecast for a new observation $X_T$ is:
$$\hat{y}_T = X_T' \hat{b}$$

And the 95% forecast interval is:
$$\hat{y}_T \pm t_{2.5\%} \cdot \sqrt{\hat{\sigma}^2 X_T'(X'X)^{-1}X_T + \hat{\sigma}^2}$$


### Exercise 7.1 — Make a forecast

Using the **pooled model** from Part 6 (or re-estimate it), predict $y$ for a new observation with $x = 5$. Compute the 95% forecast interval.


In [ ]:
# YOUR CODE HERE

# Re-estimate pooled model (on all data — ignoring the break for simplicity here)
# Or better: use the subsample 1 model, and predict for x = 5

# Hint: use model.get_prediction() from statsmodels
# X_new = sm.add_constant(np.array([[5]]))
# prediction = model1.get_prediction(X_new)
# print(prediction.summary_frame(alpha=0.05))  # 95% confidence



---
## Summary

In this lab, you practiced:

- **T-test**: Testing individual coefficients ($H_0: b_j = 0$ or $H_0: b_j = a$)
- **P-value**: The probability of observing such an extreme statistic under $H_0$
- **Size and power**: The trade-off between Type I and Type II errors
- **F-test**: Testing multiple restrictions simultaneously
- **Linear restrictions**: Testing combinations like $\alpha + \beta = 1$
- **Chow test**: Testing for structural breaks across subsamples
- **Forecasting**: Point predictions and confidence intervals

**Key takeaways:**
- Always report the significance level ($\alpha$)
- Use F-tests (not multiple T-tests) for joint hypotheses
- A non-rejection of $H_0$ does not prove $H_0$ is true — it only means there is insufficient evidence to reject it
